# AlphaGateau for Breakthrough — Standalone Demo

This notebook runs the full experiment pipeline without Slurm.  
It works on any Linux GPU: Google Colab, Kaggle, Lambda Labs, or a local machine.

**What it does:**
1. Installs dependencies and verifies JAX sees the GPU
2. Runs a 1-iteration smoke test (fast sanity check)
3. Trains all four presets (or lets you pick one)
4. Postprocesses results and generates report figures

**Time estimates (single A100/V100):**
| Stage | Time |
|---|---|
| Smoke (1 iter × 4 presets) | ~5 min |
| `gnn_8x8_scratch` (40 iters) | ~4–6 h |
| `cnn_8x8_scratch` (40 iters) | ~3–5 h |
| `gnn_5x5_pretrain` (40 iters) | ~2–4 h |
| `gnn_8x8_finetune` (30 iters) | ~2–4 h |

## 1 — Clone and install

In [ ]:
# If running on Colab or a fresh instance, clone the repo first:
# !git clone <repo-url>
# %cd alphagateau-breakthrough-v2

# Install the package and dependencies (CPU-safe base):
!pip install -e . -q

# Then install exactly ONE JAX CUDA extra matching your driver:
!pip install --upgrade 'jax[cuda12]' -q   # swap for cuda13 if needed

## 2 — Verify GPU

In [ ]:
!python scripts/show_runtime_info.py

Expected output includes `jax devices: [CudaDevice(id=0)]`.  
CPU is also fine — training will just be slower.

## 3 — Smoke test (1-iteration sanity check)

In [ ]:
!python scripts/run_smoke_suite.py

This runs one iteration of all four presets and writes preliminary figures to `report/figures/`.  
If it completes without errors the full pipeline is working.

## 4 — Run the unit tests (optional)

In [ ]:
!pytest -q -s tests/test_env.py tests/test_graph.py tests/test_training_resume.py

## 5 — Full training

Run each preset. All commands support `--resume` so you can safely restart after a kernel
crash or preemption.

> **Tip for Colab:** use background execution or run one preset at a time to avoid the
> session timeout. Save checkpoints to Google Drive by setting
> `--output-root /content/drive/MyDrive/breakthrough/experiments`.

In [ ]:
# GNN from scratch on 8x8 (~4-6 hours on a single GPU)
!python scripts/train_experiment.py gnn_8x8_scratch --resume

In [ ]:
# CNN from scratch on 8x8 (~3-5 hours)
!python scripts/train_experiment.py cnn_8x8_scratch --resume

In [ ]:
# GNN pretrain on 5x5 (~2-4 hours)
!python scripts/train_experiment.py gnn_5x5_pretrain --resume

In [ ]:
# Transfer to 8x8 from the pretrained checkpoint (~2-4 hours)
!python scripts/run_transfer_pipeline.py \
  --pretrained-checkpoint artifacts/experiments/gnn_5x5_pretrain/checkpoints/final.pkl \
  --resume

## 6 — Check experiment status at any time

In [ ]:
import json, pathlib

for run in ['gnn_8x8_scratch', 'cnn_8x8_scratch', 'gnn_5x5_pretrain', 'gnn_8x8_finetune']:
    status_path = pathlib.Path(f'artifacts/experiments/{run}/status.json')
    if status_path.exists():
        s = json.loads(status_path.read_text())
        print(f"{run}: {s['completed_iterations']}/{s['target_iterations']} iters — {s['status']}")
    else:
        print(f"{run}: not started")

## 7 — Postprocess: head-to-head evaluation and report figures

In [ ]:
!python scripts/postprocess_experiments.py \
  --gnn-scratch-dir  artifacts/experiments/gnn_8x8_scratch \
  --cnn-scratch-dir  artifacts/experiments/cnn_8x8_scratch \
  --pretrain-dir     artifacts/experiments/gnn_5x5_pretrain \
  --finetune-dir     artifacts/experiments/gnn_8x8_finetune \
  --transfer-summary artifacts/experiments/gnn_transfer_summary.json \
  --transfer-zero-shot artifacts/experiments/gnn_transfer_zero_shot.json

## 8 — Display figures inline

In [ ]:
from IPython.display import Image, display
import pathlib

for fig in ['gnn_8x8_scratch_curve.png', 'transfer_curve.png', 'encoding_visualisation.png']:
    path = pathlib.Path('report/figures') / fig
    if path.exists():
        print(f'--- {fig} ---')
        display(Image(str(path)))
    else:
        print(f'{fig}: not yet generated')

## 9 — Compile the paper (requires tectonic or pdflatex)

In [ ]:
# Install tectonic if not available:
# !curl --proto '=https' --tlsv1.2 -fsSL https://drop.tectonic.typesetting.io/install.sh | sh

!tectonic report/paper.tex
print('PDF written to report/paper.pdf')